# EMG-Based Hand Gesture Decoding: A 4-Model Comparison

**Yugandhar Kulkarni** · DRDO Research Intern · DES Pune University
arXiv cs.RO / cs.LG · Target venue: IEEE TNSRE / BioRob 2027

---

## Abstract

We present a systematic comparison of four approaches for within-subject hand gesture classification from surface EMG using the Ninapro DB2 benchmark (40 subjects, 17 gestures, Exercise B): (1) LinearSVC with time-domain features, (2) LightGBM gradient boosting on the same TD features, (3) a compact 3-layer TinyCNN-1D on raw windowed signals (52K parameters), and (4) a TinyCNN-2D on STFT log-power spectrograms (18K parameters) with SpecAugment. All models share identical within-subject evaluation protocols. The 2D spectrogram CNN outperforms the 1D CNN and tree-based baselines, approaching published state-of-the-art. We release all code for reproducible 40-subject evaluation.

**Keywords:** surface EMG · gesture recognition · STFT spectrogram · TinyCNN · LightGBM · Ninapro DB2

In [ ]:
import os, json, warnings, sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io, scipy.signal, scipy.stats
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb

warnings.filterwarnings("ignore")

kaggle_path = Path("/kaggle/input/datasets/quddusikashaf/ninapro-db2")
local_path  = Path("..") / "dataset"
IS_KAGGLE = kaggle_path.exists()
DATASET_ROOT = kaggle_path if IS_KAGGLE else local_path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment: {"Kaggle" if IS_KAGGLE else "Local"}")
print(f"Dataset: {DATASET_ROOT}")
print(f"Device : {device}")


In [ ]:
RESULTS_DIR  = Path("/kaggle/working/results") if IS_KAGGLE else DATASET_ROOT.parent / "results"
FIGURES_DIR  = Path("/kaggle/working/figures") if IS_KAGGLE else DATASET_ROOT.parent / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE=2000; N_CH=12; N_CLASSES=17
BANDPASS_LOW=10.0; BANDPASS_HIGH=500.0; FILTER_ORDER=4
WINDOW_MS=200; STEP_MS=100; MAJ_THRESH=0.7
WIN_SIZE=int(WINDOW_MS*SAMPLE_RATE/1000)
WIN_STEP=int(STEP_MS*SAMPLE_RATE/1000)

N_SUBJECTS=40; EXERCISE=1
TRAIN_REPS=[1,2,3,4]; VAL_REPS=[5]; TEST_REPS=[6]

BATCH_1D=128; LR_1D=1e-3; PAT_1D=20
BATCH_2D=64; LR_2D=1e-3; PAT_2D=20
SPEC_NPERSEG=64; SPEC_NOVERLAP=48
SPEC_NF=SPEC_NPERSEG//2+1
SPEC_NT=(WIN_SIZE-SPEC_NOVERLAP)//(SPEC_NPERSEG-SPEC_NOVERLAP)

AUG_PROB=0.4; AUG_TM=4; AUG_FM=5; AUG_AL=0.85; AUG_AH=1.15
SVM_C=1.0; SVM_ITER=10000
LGBM_EST=200; LGBM_DEPTH=8; LGBM_LR=0.05

FAST_MODE=False; FAST_N=5; FAST_EP=20
n_subj = FAST_N if FAST_MODE else N_SUBJECTS
n_ep = FAST_EP if FAST_MODE else 100
print("Config loaded.")


In [ ]:
def find_file(root, sid, cache=None):
  names=[f"S{sid}_E{EXERCISE}_A1.mat",f"s{sid}_e{EXERCISE}_a1.mat",f"S{sid:02d}_E{EXERCISE}_A1.mat"]
  for n in names:
    h=list(root.rglob(n));
    if h: return h[0]
  pool=cache if cache else list(root.rglob("*.mat"))
  for f in pool:
    if f"S{sid}" in f.stem.upper() and f"E{EXERCISE}" in f.stem.upper(): return f
  return None

def load_mat(fp):
  try:
    m=scipy.io.loadmat(str(fp),variable_names=["emg","restimulus","rerepetition"])
  except:
    import h5py; m={}
    with h5py.File(str(fp),"r") as hf:
      for k in["emg","restimulus","rerepetition"]:
        if k in hf: m[k]=np.array(hf[k]).T
  e=m["emg"].astype(np.float32)
  return e, m["restimulus"].flatten().astype(np.uint8), m["rerepetition"].flatten().astype(np.uint8)

def bandpass(emg):
  nyq=SAMPLE_RATE/2; b,a=scipy.signal.butter(FILTER_ORDER,[BANDPASS_LOW/nyq,BANDPASS_HIGH/nyq],btype="band")
  return scipy.signal.filtfilt(b,a,emg,axis=0).astype(np.float32)

def segment(emg,lab,rep):
  n=emg.shape[0]; w,l,r=[],[],[]
  for s in range(0,n-WIN_SIZE+1,WIN_STEP):
    e=s+WIN_SIZE; c=lab[s:e]; u,ct=np.unique(c,return_counts=True); mi=np.argmax(ct)
    if ct[mi]/WIN_SIZE<MAJ_THRESH: continue
    if u[mi]==0: continue
    w.append(emg[s:e,:].T); l.append(u[mi]); r.append(rep[s+WIN_SIZE//2])
  if not w: return np.empty((0,emg.shape[1],WIN_SIZE),np.float32),np.empty(0,np.uint8),np.empty(0,np.uint8)
  return np.stack(w).astype(np.float32),np.array(l,np.uint8),np.array(r,np.uint8)

def norm_per_ch(Xtr,Xva,Xte):
  mu=Xtr.mean(axis=(0,2),keepdims=True); s=Xtr.std(axis=(0,2),keepdims=True)+1e-8
  return (Xtr-mu)/s,(Xva-mu)/s,(Xte-mu)/s

def to_spec(wins):
  nw,nc,_=wins.shape; out=[]
  for c in range(nc):
    _,_,Sxx=scipy.signal.spectrogram(wins[:,c,:],fs=SAMPLE_RATE,nperseg=SPEC_NPERSEG,noverlap=SPEC_NOVERLAP,axis=-1)
    out.append(np.log1p(Sxx).astype(np.float32))
  return np.stack(out,axis=1)

def td_feat(wins):
  nw,nc,_=wins.shape; f=np.zeros((nw,nc*5),np.float32)
  for c in range(nc):
    s=wins[:,c,:]; b=c*5
    f[:,b+0]=np.mean(np.abs(s),axis=1); f[:,b+1]=np.sqrt(np.mean(s**2,axis=1))
    f[:,b+2]=np.sum(np.abs(np.diff(s,axis=1)),axis=1)
    sc=np.diff(np.sign(s),axis=1); zc=(np.abs(sc)>0)&(np.abs(s[:,:-1])>0.01); f[:,b+3]=np.sum(zc,axis=1)
    d=np.diff(s,axis=1); ssc=(np.diff(np.sign(d),axis=1)!=0)&(np.abs(d[:,:-1])>0.01); f[:,b+4]=np.sum(ssc,axis=1)
  return f

def load_sub(root,sid,cache=None):
  fp=find_file(root,sid,cache)
  if fp is None: return None
  try:
    emg,lbl,rep=load_mat(fp); emg_f=bandpass(emg); w,wl,wr=segment(emg_f,lbl,rep)
    return (w,wl,wr) if len(wl)>0 else None
  except Exception as e: print(f"S{sid} error: {e}"); return None


In [ ]:
class TinyCNN1D(nn.Module):
  def __init__(self):
    super().__init__()
    self.net=nn.Sequential(
      nn.Conv1d(12,16,5,padding=2),nn.BatchNorm1d(16),nn.ReLU(),nn.MaxPool1d(2),
      nn.Conv1d(16,32,5,padding=2),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(2),
      nn.Conv1d(32,64,3,padding=1),nn.BatchNorm1d(64),nn.ReLU(),nn.AdaptiveAvgPool1d(1),
      nn.Flatten(),nn.Dropout(0.3),nn.Linear(64,17))
  def forward(self,x): return self.net(x)

class TinyCNN2D(nn.Module):
  def __init__(self):
    super().__init__()
    self.net=nn.Sequential(
      nn.Conv2d(12,8,3,padding=1),nn.BatchNorm2d(8),nn.ReLU(),nn.MaxPool2d(2),
      nn.Conv2d(8,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
      nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
      nn.Flatten(),nn.Dropout(0.3),nn.Linear(32,17))
  def forward(self,x): return self.net(x)

class EMGDS(Dataset):
  def __init__(self,X,y): self.X=torch.from_numpy(X).float(); self.y=torch.from_numpy(y).long()
  def __len__(self): return len(self.y)
  def __getitem__(self,i): return self.X[i],self.y[i]

class SpecDS(Dataset):
  def __init__(self,S,y,aug=True):
    self.S=torch.from_numpy(S).float(); self.y=torch.from_numpy(y).long(); self.aug=aug
  def __len__(self): return len(self.y)
  def __getitem__(self,i):
    s=self.S[i].clone()
    if self.aug:
      _,nf,nt=s.shape
      if torch.rand(1).item()<AUG_PROB:
        t0=torch.randint(0,max(1,nt-1),(1,)).item(); dt=torch.randint(1,AUG_TM+1,(1,)).item()
        s[:,:,t0:min(t0+dt,nt)]=0
      if torch.rand(1).item()<AUG_PROB:
        f0=torch.randint(0,max(1,nf-1),(1,)).item(); df=torch.randint(1,AUG_FM+1,(1,)).item()
        s[:,f0:min(f0+df,nf),:]=0
      if torch.rand(1).item()<AUG_PROB:
        s=s*(AUG_AL+torch.rand(1).item()*(AUG_AH-AUG_AL))
    return s,self.y[i]

_m1=TinyCNN1D(); _m2=TinyCNN2D()
print(f"TinyCNN-1D: {sum(p.numel() for p in _m1.parameters()):,} params")
print(f"TinyCNN-2D: {sum(p.numel() for p in _m2.parameters()):,} params")
del _m1,_m2; print("Models ready.")


In [ ]:
def train_1d(Xtr,ytr,Xva,yva):
  tr=DataLoader(EMGDS(Xtr,ytr),BATCH_1D,shuffle=True)
  vl=DataLoader(EMGDS(Xva,yva),256)
  m=TinyCNN1D().to(device); opt=torch.optim.AdamW(m.parameters(),lr=LR_1D)
  sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,n_ep)
  ba=0; bs=None; ct=0
  for ep in range(n_ep):
    m.train(); cr=0; tt=0
    for xb,yb in tr:
      xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
      F.cross_entropy(m(xb),yb).backward(); opt.step()
      cr+=(m(xb).argmax(1)==yb).sum().item(); tt+=len(yb)
    sch.step()
    m.eval(); vc=0; vt=0
    with torch.no_grad():
      for xb,yb in vl:
        xb,yb=xb.to(device),yb.to(device); vc+=(m(xb).argmax(1)==yb).sum().item(); vt+=len(yb)
    va=vc/vt
    if va>ba: ba=va; bs={k:v.cpu().clone() for k,v in m.state_dict().items()}; ct=0
    else: ct+=1
    if ct>=PAT_1D: break
  m.load_state_dict(bs); return m

def train_2d(Str,ytr,Sva,yva):
  tr=DataLoader(SpecDS(Str,ytr,True),BATCH_2D,shuffle=True)
  vl=DataLoader(SpecDS(Sva,yva,False),256)
  m=TinyCNN2D().to(device); opt=torch.optim.AdamW(m.parameters(),lr=LR_2D)
  sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,n_ep)
  ba=0; bs=None; ct=0
  for ep in range(n_ep):
    m.train(); cr=0; tt=0
    for xb,yb in tr:
      xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
      F.cross_entropy(m(xb),yb).backward(); opt.step()
      cr+=(m(xb).argmax(1)==yb).sum().item(); tt+=len(yb)
    sch.step()
    m.eval(); vc=0; vt=0
    with torch.no_grad():
      for xb,yb in vl:
        xb,yb=xb.to(device),yb.to(device); vc+=(m(xb).argmax(1)==yb).sum().item(); vt+=len(yb)
    va=vc/vt
    if va>ba: ba=va; bs={k:v.cpu().clone() for k,v in m.state_dict().items()}; ct=0
    else: ct+=1
    if ct>=PAT_2D: break
  m.load_state_dict(bs); return m

@torch.no_grad()
def pred1(m,X):
  m.eval(); ld=DataLoader(EMGDS(X,np.zeros(len(X),np.int64)),256)
  return np.concatenate([m(x.to(device)).argmax(1).cpu().numpy() for x,_ in ld])

@torch.no_grad()
def pred2(m,S):
  m.eval(); ld=DataLoader(SpecDS(S,np.zeros(len(S),np.int64),False),256)
  return np.concatenate([m(x.to(device)).argmax(1).cpu().numpy() for x,_ in ld])

print("Training functions ready.")


In [ ]:
le=LabelEncoder(); le.fit(np.arange(1,N_CLASSES+1))
subject_results=[]; best_cache=None
print(f"Evaluating {n_subj} subjects | 4 models each")
print("="*100)
all_f=sorted(DATASET_ROOT.rglob("*.mat"))

for sid in range(1,n_subj+1):
  r=load_sub(DATASET_ROOT,sid,all_f)
  if r is None: print(f"S{sid:02d}: SKIP"); continue
  w,lbl,rep=r
  tr=np.isin(rep,TRAIN_REPS); va=np.isin(rep,VAL_REPS); te=np.isin(rep,TEST_REPS)
  if tr.sum()==0 or va.sum()==0 or te.sum()==0: print(f"S{sid:02d}: SKIP"); continue
  ya=le.transform(lbl); ytr=ya[tr]; yva=ya[va]; yte=ya[te]
  Xtr,Xva,Xte=norm_per_ch(w[tr],w[va],w[te])
  
  Ftr=td_feat(Xtr); Fte=td_feat(Xte)
  svm=Pipeline([("sc",StandardScaler()),("clf",LinearSVC(C=SVM_C,class_weight="balanced",max_iter=SVM_ITER,random_state=RANDOM_SEED,dual="auto"))])
  svm.fit(Ftr,ytr); yps=svm.predict(Fte)
  svm_a=accuracy_score(yte,yps); svm_f=f1_score(yte,yps,average="macro",zero_division=0)
  
  lgbm=lgb.LGBMClassifier(n_estimators=LGBM_EST,max_depth=LGBM_DEPTH,learning_rate=LGBM_LR,random_state=RANDOM_SEED,class_weight="balanced",verbose=-1)
  lgbm.fit(Ftr,ytr); ypl=lgbm.predict(Fte)
  lgb_a=accuracy_score(yte,ypl); lgb_f=f1_score(yte,ypl,average="macro",zero_division=0)
  
  m1=train_1d(Xtr,ytr,Xva,yva); yp1=pred1(m1,Xte)
  c1_a=accuracy_score(yte,yp1); c1_f=f1_score(yte,yp1,average="macro",zero_division=0); del m1
  
  Str=to_spec(Xtr); Sva=to_spec(Xva); Ste=to_spec(Xte)
  m2=train_2d(Str,ytr,Sva,yva); yp2=pred2(m2,Ste)
  c2_a=accuracy_score(yte,yp2); c2_f=f1_score(yte,yp2,average="macro",zero_division=0); del m2,Str,Sva,Ste
  
  if best_cache is None or c2_a>best_cache["a"]:
    best_cache={"s":sid,"a":c2_a,"yt":yte.copy(),"yp":yp2.copy()}
  
  subject_results.append({"id":sid,"n":int(Xtr.shape[0]),"svm_a":round(float(svm_a),4),"svm_f":round(float(svm_f),4),
    "lgb_a":round(float(lgb_a),4),"lgb_f":round(float(lgb_f),4),
    "c1_a":round(float(c1_a),4),"c1_f":round(float(c1_f),4),
    "c2_a":round(float(c2_a),4),"c2_f":round(float(c2_f),4)})
  print(f"S{sid:02d} | n={Xtr.shape[0]:4d} | SVM={svm_a:.3f} | LGBM={lgb_a:.3f} | C1D={c1_a:.3f} | C2D={c2_a:.3f}")
  if torch.cuda.is_available(): torch.cuda.empty_cache()
print("="*100)
print(f"Done. {len(subject_results)} subjects.")


In [ ]:
df=pd.DataFrame(subject_results)
for k in ["svm","lgb","c1","c2"]:
  exec(f"{k}_m=df["{k}_a"].mean(); {k}_s=df["{k}_a"].std(); {k}_fm=df["{k}_f"].mean(); {k}_fs=df["{k}_f"].std()")
print("Per-subject results:")
print(df.to_string(index=False, float_format=lambda x:f"{x:.3f}"))
print()
print("="*80)
print(f"  FINAL | SVM={svm_m:.4f}+-{svm_s:.4f} | LGBM={lgb_m:.4f}+-{lgb_s:.4f} | C1D={c1_m:.4f}+-{c1_s:.4f} | C2D={c2_m:.4f}+-{c2_s:.4f}")
print(f"  Literature: Geng2016=78.9% | Chen2022=82.87% | Zanghieri2023=82.93%")
print("="*80)


In [ ]:
fig,ax=plt.subplots(2,2,figsize=(16,12))
fig.suptitle("EMG Gesture Decoding - Ninapro DB2 (4 Models)",fontsize=13,fontweight="bold")
x=np.arange(len(df)); w=0.2
ax[0,0].bar(x-1.5*w,df["svm_a"],w,label="SVM",color="#4C72B0",alpha=0.85)
ax[0,0].bar(x-0.5*w,df["lgb_a"],w,label="LightGBM",color="#C44E52",alpha=0.85)
ax[0,0].bar(x+0.5*w,df["c1_a"],w,label="TinyCNN-1D",color="#DD8452",alpha=0.85)
ax[0,0].bar(x+1.5*w,df["c2_a"],w,label="TinyCNN-2D",color="#55A868",alpha=0.85)
for mm,cl in [(svm_m,"#4C72B0"),(lgb_m,"#C44E52"),(c1_m,"#DD8452"),(c2_m,"#55A868")]:
  ax[0,0].axhline(mm,color=cl,ls="--",alpha=0.4)
ax[0,0].set_xticks(list(range(0,len(df),5))); ax[0,0].set_ylabel("Acc"); ax[0,0].set_ylim(0,1); ax[0,0].legend(fontsize=8); ax[0,0].grid(axis="y",alpha=0.3)

ax[0,1].scatter(df["c1_a"],df["c2_a"],c="#2ca02c",alpha=0.7,s=60,edgecolors="k",lw=0.5)
lo=min(df["c1_a"].min(),df["c2_a"].min())-0.02; hi=max(df["c1_a"].max(),df["c2_a"].max())+0.02
ax[0,1].plot([lo,hi],[lo,hi],"k--",lw=1.2,alpha=0.5)
ax[0,1].set_xlabel("C1D Acc"); ax[0,1].set_ylabel("C2D Acc"); ax[0,1].grid(alpha=0.3)
n_imp=(df["c2_a"]>df["c1_a"]).sum()
ax[0,1].text(0.05,0.92,f"C2D better in {n_imp}/{len(df)}",transform=ax[0,1].transAxes,fontsize=9,color="#2ca02c")

names=["SVM","LightGBM","C1D","C2D","Geng16","Chen22","Zanghieri23"]
means=[svm_m*100,lgb_m*100,c1_m*100,c2_m*100,78.90,82.87,82.93]
stds=[svm_s*100,lgb_s*100,c1_s*100,c2_s*100,0,0,0]
cs=["#4C72B0","#C44E52","#DD8452","#55A868","#8da0cb","#8da0cb","#8da0cb"]
b=ax[1,0].bar(range(len(names)),means,yerr=stds,color=cs,alpha=0.85,capsize=5,edgecolor="white")
ax[1,0].set_xticks(range(len(names))); ax[1,0].set_xticklabels(names,fontsize=8)
ax[1,0].set_ylabel("Acc (%)"); ax[1,0].set_ylim(40,100); ax[1,0].grid(axis="y",alpha=0.3)
for br,v in zip(b,means):
  ax[1,0].text(br.get_x()+br.get_width()/2,br.get_height()+0.5,f"{v:.1f}%",ha="center",va="bottom",fontsize=7,fontweight="bold")

cm=confusion_matrix(best_cache["yt"],best_cache["yp"])
cmn=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-9)
ax[1,1].imshow(cmn,cmap="Blues",vmin=0,vmax=1,aspect="auto"); plt.colorbar(ax[1,1].images[0],ax=ax[1,1],shrink=0.8)
ax[1,1].set_xlabel("Pred"); ax[1,1].set_ylabel("True")
ax[1,1].set_title(f"Best S{best_cache["s"]:02d} acc={best_cache["a"]:.3f}")
plt.tight_layout()
plt.savefig(FIGURES_DIR/"results_4model.png",dpi=150,bbox_inches="tight")
plt.show()
print("Figure saved.")


In [ ]:
from scipy import stats
t,p=stats.ttest_rel(df["c2_a"],df["c1_a"])
cd=(df["c2_a"]-df["c1_a"]).mean()/(df["c2_a"]-df["c1_a"]).std()
t2,p2=stats.ttest_rel(df["lgb_a"],df["svm_a"])
print("Stats:")
print(f"  C2D vs C1D: t={t:.4f} p={p:.4f} d={cd:.4f}")
print(f"  LGB vs SVM: t={t2:.4f} p={p2:.4f}")
n2v1=(df["c2_a"]>df["c1_a"]).sum()
lvs=(df["lgb_a"]>df["svm_a"]).sum()
print(f"  C2D>C1D: {n2v1}/{len(df)} ({n2v1/len(df)*100:.1f}%)")
print(f"  LGB>SVM: {lvs}/{len(df)} ({lvs/len(df)*100:.1f}%)")


In [ ]:
from scipy import stats as _st
t1,p1=_st.ttest_rel(df["c2_a"],df["c1_a"])
final={
  "experiment":"EMG Gesture Decoding - Ninapro DB2 Ex1 (4-Model)",
  "dataset":{"name":"Ninapro DB2","doi":"10.1038/sdata.2014.53","n_subjects":N_SUBJECTS,"n_classes":N_CLASSES,"sample_rate":SAMPLE_RATE},
  "svm":{"model":"LinearSVC C=1.0","mean_acc":round(float(svm_m),4),"std_acc":round(float(svm_s),4),"mean_f1":round(float(svm_fm),4),"std_f1":round(float(svm_fs),4)},
  "lightgbm":{"model":f"LGBM n_est={LGBM_EST}","mean_acc":round(float(lgb_m),4),"std_acc":round(float(lgb_s),4),"mean_f1":round(float(lgb_fm),4),"std_f1":round(float(lgb_fs),4)},
  "tinycnn1d":{"model":"TinyCNN-1D 16-32-64","params":51857,"mean_acc":round(float(c1_m),4),"std_acc":round(float(c1_s),4),"mean_f1":round(float(c1_fm),4),"std_f1":round(float(c1_fs),4)},
  "tinycnn2d":{"model":"TinyCNN-2D 8-16-32","params":17745,"mean_acc":round(float(c2_m),4),"std_acc":round(float(c2_s),4),"mean_f1":round(float(c2_fm),4),"std_f1":round(float(c2_fs),4)},
  "paired_ttest":{"c2d_vs_c1d":{"t":round(float(t1),4),"p":round(float(p1),6)}},
  "per_subject":subject_results,
  "device":str(device),
  "timestamp":datetime.now().isoformat(),
  "seed":RANDOM_SEED,
}
with open(RESULTS_DIR/"final_results.json","w") as f: json.dump(final,f,indent=2)
print(f"Saved -> {RESULTS_DIR/"final_results.json"}")


## Limitations and Future Work

**Limitations:** All DB2 subjects are healthy; results may not transfer to amputees. Rep-based split evaluates single session only — cross-day drift is not tested. Fixed electrode placement — no robustness to electrode shift.

**Future Work:** Cross-subject pretrain + fine-tune, INT8 quantization for embedded deployment (<100ms), contrastive pre-training across Ninapro DB1/DB2/DB5, causal streaming with LSTM/Conformer on STFT frames.

## Citation

```bibtex
@misc{kulkarni2026emgstft,
  title  = {{EMG}-Based Hand Gesture Decoding: A Controlled Comparison of
             Raw-Signal and Time-Frequency Input Representations},
  author = {Kulkarni, Yugandhar},
  year   = {2026},
  note   = {arXiv preprint arXiv:2026.XXXXX [cs.RO]}
}
```

---
*Yugandhar Kulkarni · DRDO Research Intern · DES Pune University*